# 04 — Validation and extra experiments

Training-only validation for every selected value (d, m), then E4 training size, optional E5 random split, E6 regularised Mahalanobis.

Run notebooks 02 and 03 first: test accuracies at the selected values are read from their saved arrays.

In [ ]:
import sys, pathlib
sys.path.append(str(pathlib.Path.cwd().parent))
%load_ext autoreload
%autoreload 2

In [ ]:
import warnings

import numpy as np

from src.config import OFFSETS, RESULTS_DIR
from src.data import load_coil20, make_split
from src.evaluate import CLASSIFIERS, lda_scorer, pca_scorer, select_by_validation

ARRAYS = RESULTS_DIR / "arrays"
names = list(CLASSIFIERS)

In [ ]:
X, y, pose = load_coil20()
pca = np.load(ARRAYS / "pca_acc.npz")
lda = np.load(ARRAYS / "lda_m_sweep.npz")

## Training-only validation (section 8.1)

For each split, leave-one-pose-out inside the 240 training images: 12 folds, each fitting on 220 images and validating on the 20 at the held-out pose. Test indices are never passed in. The selected value is then looked up on the test curve.

- Inside a fold N = 220, so PCA d ≤ 219, min-Mahal d ≤ N − c = 200, and LDA m ≤ 200. Candidates undefined in any fold are skipped.
- Ties go to the smallest d (or m).
- The held-out pose is 30° from its nearest remaining training pose, whereas test images are at most 15° away, so validation accuracy is pessimistic. It is used for ranking only.

### PCA: select d per classifier

In [ ]:
d_pca = [int(d) for d in pca["d_list"] if d <= 219]
pca_d = np.zeros((len(OFFSETS), len(names)), dtype=int)
pca_val = np.zeros(pca_d.shape)
pca_test = np.zeros(pca_d.shape)
for k, off in enumerate(OFFSETS):
    tr, _ = make_split(pose, off)
    for i, predict in enumerate(CLASSIFIERS.values()):
        d, scores = select_by_validation(X, y, pose, tr, d_pca, pca_scorer(predict, d_pca))
        pca_d[k, i], pca_val[k, i] = d, np.nanmax(scores)
        pca_test[k, i] = pca["acc"][k, i, np.flatnonzero(pca["d_list"] == d)[0]]
with warnings.catch_warnings():
    warnings.simplefilter("ignore", RuntimeWarning)   # all-nan min-Mahal columns above the cap
    best = np.nanmax(np.nanmean(pca["acc"], axis=0), axis=1)
for i, n in enumerate(names):
    print(f"{n:11s} d per split {pca_d[:, i]}  val {pca_val[:, i].mean():.4f}  "
          f"test {pca_test[:, i].mean():.4f}±{pca_test[:, i].std():.4f}  (best on test curve {best[i]:.4f})")

### PCA → LDA: select (m, d) jointly per classifier

m from the sweep in notebook 03, excluding 220 (above the fold cap of 200).

In [ ]:
m_list = [int(m) for m in lda["m_list"] if m <= 200]
d_lda = [int(d) for d in lda["d_list"]]
lda_m = np.zeros((len(OFFSETS), len(names)), dtype=int)
lda_d = np.zeros_like(lda_m)
lda_val = np.zeros(lda_m.shape)
lda_test = np.zeros(lda_m.shape)
for k, off in enumerate(OFFSETS):
    tr, _ = make_split(pose, off)
    for i, predict in enumerate(CLASSIFIERS.values()):
        candidates, score_fn = lda_scorer(predict, m_list, d_lda)
        (mp, d), scores = select_by_validation(X, y, pose, tr, candidates, score_fn)
        lda_m[k, i], lda_d[k, i], lda_val[k, i] = mp, d, np.nanmax(scores)
        a = np.flatnonzero(lda["m_list"] == mp)[0]
        lda_test[k, i] = lda["acc"][a, k, i, np.flatnonzero(lda["d_list"] == d)[0]]
for i, n in enumerate(names):
    print(f"{n:11s} m {lda_m[:, i]}  d {lda_d[:, i]}  val {lda_val[:, i].mean():.4f}  "
          f"test {lda_test[:, i].mean():.4f}±{lda_test[:, i].std():.4f}")

In [ ]:
np.savez(ARRAYS / "selected.npz", classifiers=names,
         pca_d=pca_d, pca_val=pca_val, pca_test=pca_test,
         lda_m=lda_m, lda_d=lda_d, lda_val=lda_val, lda_test=lda_test)